# Descriptors comparison

In [ ]:
# =============================================================================
# GENERATE FINGERPRINTS + DESCRIPTORS -> SAVE FULL FEATURE MATRIX TO PARQUET
# =============================================================================

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors
from rdkit.Chem.Pharm2D import Gobbi_Pharm2D, Generate
from rdkit.ML.Descriptors import MoleculeDescriptors
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

# -----------------------------------------------------------------------------
# 1.LOAD DATA
# -----------------------------------------------------------------------------

print("Loading dataset...")

df = pd.read_excel("wildtype_egfr.xlsx")

# Validate required columns
assert "SMILES" in df.columns, "SMILES column missing!"
assert "Bioactivity" in df.columns, "Bioactivity column missing!"
assert "pIC50 (M)" in df.columns, "pIC50 (M) column missing!"

# Create label
df["Bioactivity"] = df["Bioactivity"].str.lower()
df["y"] = df["Bioactivity"].map({"less potent": 0, "potent": 1})

# Canonicalize SMILES
def canonicalize(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        return Chem.MolToSmiles(mol, canonical=True) if mol else None
    except:
        return None

df["SMILES"] = df["SMILES"].apply(canonicalize)

# Remove invalid molecules
df.dropna(subset=["SMILES", "y", "pIC50 (M)"], inplace=True)

print("Remaining molecules:", len(df))

# Generate RDKit molecule objects
mols = [Chem.MolFromSmiles(s) for s in df["SMILES"]]

# -----------------------------------------------------------------------------
# 2. GENERATE ECFP6 (2048 bits)
# -----------------------------------------------------------------------------

print("Generating ECFP6...")

ecfp = [
    list(AllChem.GetMorganFingerprintAsBitVect(m, radius=3, nBits=2048))
    for m in mols
]

X_ecfp = pd.DataFrame(ecfp, columns=[f"ECFP_{i}" for i in range(2048)])

# -----------------------------------------------------------------------------
# 3. GENERATE Pharm2D
# -----------------------------------------------------------------------------

print("Generating Pharm2D...")

factory = Gobbi_Pharm2D.factory
pharm = []

for m in mols:
    try:
        pharm.append(list(Generate.Gen2DFingerprint(m, factory)))
    except:
        pharm.append([np.nan] * factory.GetSigSize())

X_pharm = pd.DataFrame(
    pharm,
    columns=[f"Pharm_{i}" for i in range(factory.GetSigSize())]
)

# -----------------------------------------------------------------------------
# 4. GENERATE RDKit DESCRIPTORS
# -----------------------------------------------------------------------------

print("Generating RDKit descriptors...")

desc_names = [name for name, _ in Descriptors._descList]
calc = MoleculeDescriptors.MolecularDescriptorCalculator(desc_names)

rd_desc = []

for m in mols:
    try:
        rd_desc.append(list(calc.CalcDescriptors(m)))
    except:
        rd_desc.append([np.nan] * len(desc_names))

X_rdkit = pd.DataFrame(
    rd_desc,
    columns=[f"RDKit_{name}" for name in desc_names]
)

# -----------------------------------------------------------------------------
# 5. MERGE ALL FEATURES
# -----------------------------------------------------------------------------

print("Merging feature matrices...")

X_all = pd.concat([X_ecfp, X_pharm, X_rdkit], axis=1)

# Add SMILES + pIC50 + label
X_all["SMILES"] = df["SMILES"].values
X_all["pIC50 (M)"] = df["pIC50 (M)"].values
X_all["y"] = df["y"].values

# Reorder columns
cols = ["SMILES", "pIC50 (M)", "y"] + [
    c for c in X_all.columns if c not in ["SMILES", "pIC50 (M)", "y"]
]

X_all = X_all[cols]

print("Final shape:", X_all.shape)

# -----------------------------------------------------------------------------
# 6. SAVE TO PARQUET
# -----------------------------------------------------------------------------

output_file = "wildtype egfr_features.parquet"

print("Saving to Parquet...")
X_all.to_parquet(output_file, index=False, engine="pyarrow")

print("Saved successfully:", output_file)

In [29]:
# =============================================================================
# Load Precomputed Features → 7 Combinations → Variance → Correlation → Mutual Info → RF / XGB / LGBM
# =============================================================================

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.feature_selection import mutual_info_classif
from collections import Counter

import lightgbm as lgb
import xgboost as xgb
import joblib
import random
import os

# -------------------------------------------------------------------------
# SEED
# -------------------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# -------------------------------------------------------------------------
# 1. LOAD FEATURES
# -------------------------------------------------------------------------
print("Loading feature matrix...")
X_all_df = pd.read_parquet("wildtype egfr_features.parquet")
print("Loaded shape:", X_all_df.shape)

y = X_all_df["y"].values
X_features = X_all_df.drop(columns=["SMILES", "y"])

# Freeze feature order
feature_columns = list(X_features.columns)

# Split feature types
X_ecfp  = X_features.filter(regex="^ECFP_")
X_pharm = X_features.filter(regex="^Pharm_")
X_rdkit = X_features.filter(regex="^RDKit_")

# -------------------------------------------------------------------------
# 2. DEFINE SETS
# -------------------------------------------------------------------------
fingerprint_sets = {
    "ECFP": (X_ecfp, None),
    "Pharm2D": (X_pharm, None),
    "RDKit": (None, X_rdkit),
    "ECFP+Pharm2D": (pd.concat([X_ecfp, X_pharm], axis=1), None),
    "ECFP+RDKit": (X_ecfp, X_rdkit),
    "Pharm2D+RDKit": (X_pharm, X_rdkit),
    "ALL": (pd.concat([X_ecfp, X_pharm], axis=1), X_rdkit)
}

# -------------------------------------------------------------------------
# 3. SPLIT (FROZEN)
# -------------------------------------------------------------------------
X_train_base, X_test_base, y_train, y_test = train_test_split(
    X_features,
    y,
    test_size=0.2,
    stratify=y,
    random_state=SEED
)

train_idx = X_train_base.index
test_idx  = X_test_base.index

# -------------------------------------------------------------------------
# 4. PIPELINE
# -------------------------------------------------------------------------
results = {}
os.makedirs("saved_models", exist_ok=True)

for name, (X_bin_all, X_cont_all) in fingerprint_sets.items():

    print(f"\n================ {name} ================")

    # -------------------------------
    # SPLIT FEATURES
    # -------------------------------
    if X_bin_all is not None:
        X_train_bin = X_bin_all.loc[train_idx].copy()
        X_test_bin  = X_bin_all.loc[test_idx].copy()
        bin_columns = list(X_train_bin.columns)
    else:
        X_train_bin = X_test_bin = None
        bin_columns = None

    if X_cont_all is not None:
        X_train_cont = X_cont_all.loc[train_idx].copy()
        X_test_cont  = X_cont_all.loc[test_idx].copy()

        # Drop all-NaN columns (train based)
        X_train_cont = X_train_cont.dropna(axis=1, how="all")
        X_test_cont  = X_test_cont[X_train_cont.columns]

        cont_columns_initial = list(X_train_cont.columns)
    else:
        X_train_cont = X_test_cont = None
        cont_columns_initial = None

    # -------------------------------
    # BEFORE FEATURE SELECTION
    # -------------------------------
    if X_train_bin is not None and X_train_cont is not None:
        before_shape = X_train_bin.shape[1] + X_train_cont.shape[1]
    elif X_train_bin is not None:
        before_shape = X_train_bin.shape[1]
    else:
        before_shape = X_train_cont.shape[1]

    print("Before Feature Selection:", before_shape)

    # -------------------------------
    # Variance (Binary)
    # -------------------------------
    if X_train_bin is not None:
        vt_bin = VarianceThreshold(0.05)
        X_train_bin = vt_bin.fit_transform(X_train_bin)
        X_test_bin  = vt_bin.transform(X_test_bin)
        print("After Binary Variance:", X_train_bin.shape[1])
    else:
        vt_bin = None

    # -------------------------------
    # Continuous Processing
    # -------------------------------
    if X_train_cont is not None:

        vt_cont = VarianceThreshold(0.0)
        X_train_cont = vt_cont.fit_transform(X_train_cont)
        X_test_cont  = vt_cont.transform(X_test_cont)

        cont_after_vt = np.array(cont_columns_initial)[vt_cont.get_support()]

        X_train_df = pd.DataFrame(X_train_cont, columns=cont_after_vt)

        corr = X_train_df.corr().abs()
        upper = corr.where(np.triu(np.ones(corr.shape), 1).astype(bool))
        corr_drop_cols = [c for c in upper.columns if any(upper[c] > 0.9)]

        X_train_df = X_train_df.drop(columns=corr_drop_cols)
        X_test_df  = pd.DataFrame(X_test_cont, columns=cont_after_vt)\
                        .drop(columns=corr_drop_cols)

        print("After Continuous Variance + Correlation:", X_train_df.shape[1])

        cont_final_cols = list(X_train_df.columns)

        imputer = SimpleImputer(strategy="median")
        scaler  = StandardScaler()

        X_train_cont = scaler.fit_transform(imputer.fit_transform(X_train_df))
        X_test_cont  = scaler.transform(imputer.transform(X_test_df))

    else:
        vt_cont = None
        imputer = None
        scaler  = None
        corr_drop_cols = None
        cont_after_vt = None
        cont_final_cols = None

    # -------------------------------
    # Merge
    # -------------------------------
    if X_train_bin is not None and X_train_cont is not None:
        X_train_final = np.hstack([X_train_bin, X_train_cont])
        X_test_final  = np.hstack([X_test_bin,  X_test_cont])
    elif X_train_bin is not None:
        X_train_final = X_train_bin
        X_test_final  = X_test_bin
    else:
        X_train_final = X_train_cont
        X_test_final  = X_test_cont

    print("Before Mutual Information:", X_train_final.shape[1])

    # -------------------------------
    # Mutual Information
    # -------------------------------
    mi = mutual_info_classif(X_train_final, y_train, random_state=SEED)
    mi_threshold = np.percentile(mi, 50)
    selected_idx = np.where(mi >= mi_threshold)[0]

    X_train_final = X_train_final[:, selected_idx]
    X_test_final  = X_test_final[:, selected_idx]

    print("After Mutual Information:", X_train_final.shape[1])

    # -------------------------------
    # Imbalance
    # -------------------------------
    counter = Counter(y_train)
    scale_pos_weight = counter[0] / counter[1]

    model_scores = {}

    # RF
    rf_model = RandomForestClassifier(
        n_estimators=500,
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1
    )
    rf_model.fit(X_train_final, y_train)
    rf_f1 = f1_score(y_test, rf_model.predict(X_test_final))
    print(f"RF TEST F1: {rf_f1:.4f}")
    model_scores["RF"] = rf_f1

    # XGB
    xgb_model = xgb.XGBClassifier(
        n_estimators=500,
        random_state=SEED,
        n_jobs=-1,
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight,
        tree_method="hist"
    )
    xgb_model.fit(X_train_final, y_train)
    xgb_f1 = f1_score(y_test, xgb_model.predict(X_test_final))
    print(f"XGB TEST F1: {xgb_f1:.4f}")
    model_scores["XGB"] = xgb_f1

    # LGBM
    lgb_model = lgb.LGBMClassifier(
        n_estimators=500,
        random_state=SEED,
        n_jobs=-1,
        scale_pos_weight=scale_pos_weight,
        verbose=-1
    )
    lgb_model.fit(X_train_final, y_train)
    lgb_f1 = f1_score(y_test, lgb_model.predict(X_test_final))
    print(f"LGBM TEST F1: {lgb_f1:.4f}")
    model_scores["LGBM"] = lgb_f1

    # -----------------------------------------------------------------
    # SELECT BEST MODEL
    # -----------------------------------------------------------------
    best_model_name = max(model_scores, key=model_scores.get)
    best_score = model_scores[best_model_name]

    print(f"BEST MODEL ({name}): {best_model_name} | TEST F1 = {best_score:.4f}")

    results[name] = best_score

    # -----------------------------------------------------------------
    # SAVE EVERYTHING (FULL STATE)
    # -----------------------------------------------------------------
    joblib.dump({
        "seed": SEED,
        "feature_columns": feature_columns,
        "train_idx": train_idx,
        "test_idx": test_idx,

        "bin_columns": bin_columns,
        "cont_columns_initial": cont_columns_initial,
        "cont_after_vt": cont_after_vt,
        "cont_final_cols": cont_final_cols,
        "corr_drop_cols": corr_drop_cols,

        "vt_bin": vt_bin,
        "vt_cont": vt_cont,
        "imputer": imputer,
        "scaler": scaler,
        "selected_idx": selected_idx,

        "rf": rf_model,
        "xgb": xgb_model,
        "lgbm": lgb_model,
        "best_model_name": best_model_name,
        "best_score": best_score

    }, f"saved_models/{name}.pkl")

    print(f"Model saved → saved_models/{name}.pkl")

# -------------------------------------------------------------------------
# FINAL RANKING
# -------------------------------------------------------------------------
print("\n================ FINAL RANKING ================")

for rank, (name, score) in enumerate(
        sorted(results.items(), key=lambda x: x[1], reverse=True), 1):
    print(f"{rank}. {name:20s} | BEST TEST F1 = {score:.4f}")

Loading feature matrix...
Loaded shape: (5276, 42240)

================ ECFP ================
Before Feature Selection: 2048
After Binary Variance: 290
Before Mutual Information: 290
After Mutual Information: 145
RF TEST F1: 0.6824
XGB TEST F1: 0.7044
LGBM TEST F1: 0.7228
BEST MODEL (ECFP): LGBM | TEST F1 = 0.7228
Model saved → saved_models/ECFP.pkl

================ Pharm2D ================
Before Feature Selection: 39972
After Binary Variance: 2743
Before Mutual Information: 2743
After Mutual Information: 1372
RF TEST F1: 0.6601
XGB TEST F1: 0.6998
LGBM TEST F1: 0.7050
BEST MODEL (Pharm2D): LGBM | TEST F1 = 0.7050
Model saved → saved_models/Pharm2D.pkl

================ RDKit ================
Before Feature Selection: 217
After Continuous Variance + Correlation: 164
Before Mutual Information: 164
After Mutual Information: 82
RF TEST F1: 0.6449
XGB TEST F1: 0.7002
LGBM TEST F1: 0.7002
BEST MODEL (RDKit): XGB | TEST F1 = 0.7002
Model saved → saved_models/RDKit.pkl

================ ECF

# Machine Learning Building and Development

## Model 1

In [16]:
# =============================================================================
# CELL 1: Load data → Split
# =============================================================================

import pandas as pd
from sklearn.model_selection import train_test_split

# -------------------------------------------------------------------------
# 1. LOAD FEATURE FILE
# -------------------------------------------------------------------------

df = pd.read_parquet("wildtype egfr_features.parquet")
print(f"Total samples loaded: {len(df)}")

assert "y" in df.columns, "Column y not found!"
assert "pIC50 (M)" in df.columns, "Column pIC50 (M) not found!"

df.columns = [
    col.replace("ECFP6_", "ECFP_") if col.startswith("ECFP6_") else col
    for col in df.columns
]

df.columns = [
    col.replace("Pharm2D_", "Pharm_") if col.startswith("Pharm2D_") else col
    for col in df.columns
]

# -------------------------------------------------------------------------
# 2. SELECT FINGERPRINT COLUMNS
# -------------------------------------------------------------------------

ecfp_cols = [c for c in df.columns if c.startswith("ECFP_")]
pharm_cols = [c for c in df.columns if c.startswith("Pharm_")]

print(f"ECFP features   : {len(ecfp_cols)}")
print(f"Pharm2D features: {len(pharm_cols)}")

feature_cols = ecfp_cols + pharm_cols

X = df[feature_cols].copy()
y = df["y"].copy()

print(f"Total fingerprint features used: {X.shape[1]}")

# -------------------------------------------------------------------------
# 3. TRAIN / VAL / TEST SPLIT (70 / 10 / 20)
# -------------------------------------------------------------------------

X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp,
    test_size=0.125,  # 0.125 × 0.8 = 0.10
    random_state=42,
    stratify=y_tmp
)

print(f"Train size      : {X_train.shape}")
print(f"Validation size : {X_val.shape}")
print(f"Test size       : {X_test.shape}")

print("Cell 1 finished: Data ready.")

Total samples loaded: 5276
ECFP features   : 2048
Pharm2D features: 39972
Total fingerprint features used: 42020
Train size      : (3692, 42020)
Validation size : (528, 42020)
Test size       : (1056, 42020)
Cell 1 finished: Data ready.


In [17]:
# =============================================================================
# CELL 2: FEATURE SELECTION: Variance → Mutual Information
# =============================================================================

from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import mutual_info_classif
import numpy as np

# -------------------------------------------------------------------------
# 1. VARIANCE THRESHOLD (binary fingerprints)
# -------------------------------------------------------------------------

print("Before VarianceThreshold:", X_train.shape)

vt = VarianceThreshold(threshold=0.05)
X_train_vt = vt.fit_transform(X_train)
X_val_vt   = vt.transform(X_val)
X_test_vt  = vt.transform(X_test)

print("After VarianceThreshold:", X_train_vt.shape)

# -------------------------------------------------------------------------
# 2. MUTUAL INFORMATION
# -------------------------------------------------------------------------

mi = mutual_info_classif(
    X_train_vt,
    y_train,
    random_state=42
)

# Keep top 50%
mi_threshold = np.percentile(mi, 50)
selected_idx = np.where(mi >= mi_threshold)[0]

X_train_final = X_train_vt[:, selected_idx]
X_val_final   = X_val_vt[:, selected_idx]
X_test_final  = X_test_vt[:, selected_idx]

print("After Mutual Information:", X_train_final.shape)

print("Cell 2 finished: feature selection completed")


Before VarianceThreshold:

 (3692, 42020)
After VarianceThreshold: (3692, 3017)
After Mutual Information: (3692, 1509)
Cell 2 finished: feature selection completed


In [ ]:
# =============================================================================
# CELL 3: OPTUNA → MODEL COMPARISON (RF / XGB / LGBM)
# =============================================================================

import optuna
import warnings
warnings.filterwarnings("ignore")

from collections import Counter
from sklearn.metrics import f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier

import xgboost as xgb
import lightgbm as lgb
import numpy as np

import random
import numpy as np
import os

random.seed(42)
np.random.seed(42)
os.environ["PYTHONHASHSEED"] = "42"

import joblib

# -------------------------------------------------------------------------
# 0. HANDLE IMBALANCE (TRAIN SET ONLY)
# -------------------------------------------------------------------------
counter = Counter(y_train)
n_neg = counter[0]
n_pos = counter[1]
scale_pos_weight = n_neg / n_pos

print(f"Class distribution (train): {counter}")
print(f"scale_pos_weight (XGB/LGBM): {scale_pos_weight:.3f}")

# -------------------------------------------------------------------------
# 1. OBJECTIVE FUNCTION (TUNING ON VALIDATION SET)
# -------------------------------------------------------------------------
def objective(trial, model_name):

    if model_name == "RF":
        model = RandomForestClassifier(
            n_estimators=trial.suggest_int("n_estimators", 200, 800),
            max_depth=trial.suggest_int("max_depth", 5, 30),
            min_samples_split=trial.suggest_int("min_samples_split", 2, 10),
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 5),
            max_features=trial.suggest_categorical("max_features", ["sqrt", "log2"]),
            class_weight="balanced",
            n_jobs=1,
            random_state=42
        )
        model.fit(X_train_final, y_train)

    elif model_name == "XGB":
        model = xgb.XGBClassifier(
            n_estimators=trial.suggest_int("n_estimators", 200, 500),
            max_depth=trial.suggest_int("max_depth", 4, 8),
            learning_rate=trial.suggest_float("learning_rate", 0.03, 0.15, log=True),
            subsample=trial.suggest_float("subsample", 0.7, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 0.8),
            tree_method="hist",
            eval_metric="logloss",
            scale_pos_weight=scale_pos_weight,
            n_jobs=1,
            random_state=42,
            seed=42
        )
        model.fit(X_train_final, y_train)

    elif model_name == "LGBM":
        model = lgb.LGBMClassifier(
            n_estimators=trial.suggest_int("n_estimators", 300, 1000),
            max_depth=trial.suggest_int("max_depth", -1, 20),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            num_leaves=trial.suggest_int("num_leaves", 31, 256),
            subsample=trial.suggest_float("subsample", 0.6, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
            scale_pos_weight=scale_pos_weight, # Using weight instead of 'balanced' for consistency
            n_jobs=1,
            random_state=42,
            feature_fraction_seed=42,
            bagging_seed=42,
            data_random_seed=42,
            verbose=-1
        )
        model.fit(
            X_train_final, y_train,
            eval_set=[(X_val_final, y_val)],
            eval_metric="f1",
            callbacks=[lgb.early_stopping(50, verbose=False)]
        )
    else:
        raise ValueError("Unknown model")

    # VALIDATION SCORE ONLY
    y_val_pred = model.predict(X_val_final)
    return f1_score(y_val, y_val_pred)

# -------------------------------------------------------------------------
# 2. RUN OPTUNA
# -------------------------------------------------------------------------
models = ["RF", "XGB", "LGBM"]
results = {}

for model_name in models:
    print(f"\n Optimizing {model_name}...")
    study = optuna.create_study(direction="maximize")
    study.optimize(lambda trial: objective(trial, model_name), n_trials=30)

    results[model_name] = {
        "best_val_f1": study.best_value,
        "best_params": study.best_params
    }
    print(f"Best VAL F1 ({model_name}): {study.best_value:.4f}")

# -------------------------------------------------------------------------
# 3. MODEL COMPARISON (BASED ON VAL SCORE)
# -------------------------------------------------------------------------
best_model_name = max(results, key=lambda k: results[k]["best_val_f1"])
best_params = results[best_model_name]["best_params"]

print("\n================ MODEL COMPARISON (VAL) ================")
for m, res in results.items():
    print(f"{m:6s} | VAL F1 = {res['best_val_f1']:.4f}")

print(f"\n WINNER: {best_model_name}")

# -------------------------------------------------------------------------
# 4. FINAL RETRAIN (TRAIN + VAL) AND TEST EVALUATION
# -------------------------------------------------------------------------
print("\n Retraining best model on TRAIN + VAL...")

X_trainval = np.vstack([X_train_final, X_val_final])
y_trainval = np.hstack([y_train, y_val])

# Recalculate scale_pos_weight on TRAIN + VAL
counter_trainval = Counter(y_trainval)
n_neg_tv = counter_trainval[0]
n_pos_tv = counter_trainval[1]
scale_pos_weight_tv = n_neg_tv / n_pos_tv

print(f"Class distribution (train+val): {counter_trainval}")
print(f"New scale_pos_weight: {scale_pos_weight_tv:.3f}")


if best_model_name == "RF":
    final_model = RandomForestClassifier(
        **best_params,
        class_weight="balanced",
        n_jobs=-1,
        random_state=42
    )
elif best_model_name == "XGB":
    final_model = xgb.XGBClassifier(
        **best_params,
        scale_pos_weight=scale_pos_weight_tv,
        n_jobs=-1,
        random_state=42
    )
elif best_model_name == "LGBM":
    final_model = lgb.LGBMClassifier(
        **best_params,
        scale_pos_weight=scale_pos_weight_tv,
        n_jobs=-1,
        random_state=42,
        verbose=-1
    )

final_model.fit(X_trainval, y_trainval)

# Save the final model along with feature selection objects
model_package = {
    "model": final_model,
    "model_name": best_model_name,
    "best_params": best_params,
    "variance_selector": vt,
    "selected_idx": selected_idx
}

joblib.dump(model_package, "model_1.pkl")

print("\nModel saved as model_1.pkl")

# THE ONE AND ONLY TEST EVALUATION
y_test_pred = final_model.predict(X_test_final)
test_f1 = f1_score(y_test, y_test_pred)

print(f"\n FINAL TEST F1 ({best_model_name}): {test_f1:.4f}")
print("\n=== Classification Report (TEST SET) ===")
print(classification_report(y_test, y_test_pred, target_names=["Less potent", "Potent"], digits=4))

Class distribution (train): Counter({0: 2576, 1: 1116})
scale_pos_weight (XGB/LGBM): 2.308

 Optimizing RF...


[I 2026-02-23 18:47:20,165] A new study created in memory with name: no-name-957db74e-91b4-4531-b801-ee54551b978d
[I 2026-02-23 18:47:31,040] Trial 0 finished with value: 0.6888217522658611 and parameters: {'n_estimators': 605, 'max_depth': 18, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'log2'}. Best is trial 0 with value: 0.6888217522658611.
[I 2026-02-23 18:47:47,514] Trial 1 finished with value: 0.7040498442367601 and parameters: {'n_estimators': 460, 'max_depth': 25, 'min_samples_split': 7, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.7040498442367601.
[I 2026-02-23 18:48:11,515] Trial 2 finished with value: 0.6949152542372882 and parameters: {'n_estimators': 780, 'max_depth': 15, 'min_samples_split': 9, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.7040498442367601.
[I 2026-02-23 18:48:16,461] Trial 3 finished with value: 0.6888888888888889 and parameters: {'n_estimators': 342, 'max_depth': 23, 'm

Best VAL F1 (RF): 0.7130

 Optimizing XGB...


[I 2026-02-23 18:55:48,306] Trial 0 finished with value: 0.6978193146417445 and parameters: {'n_estimators': 470, 'max_depth': 5, 'learning_rate': 0.09326018567958562, 'subsample': 0.99443687333358, 'colsample_bytree': 0.7967400397570001}. Best is trial 0 with value: 0.6978193146417445.
[I 2026-02-23 18:56:01,721] Trial 1 finished with value: 0.7049180327868853 and parameters: {'n_estimators': 235, 'max_depth': 5, 'learning_rate': 0.03186762361593829, 'subsample': 0.7036812367725979, 'colsample_bytree': 0.7592214669067435}. Best is trial 1 with value: 0.7049180327868853.
[I 2026-02-23 18:56:13,555] Trial 2 finished with value: 0.7048192771084337 and parameters: {'n_estimators': 382, 'max_depth': 4, 'learning_rate': 0.12494255161675083, 'subsample': 0.9914569446283593, 'colsample_bytree': 0.5950584926572476}. Best is trial 1 with value: 0.7049180327868853.
[I 2026-02-23 18:56:35,112] Trial 3 finished with value: 0.7317073170731707 and parameters: {'n_estimators': 493, 'max_depth': 6, 'l

Best VAL F1 (XGB): 0.7317

 Optimizing LGBM...


[I 2026-02-23 19:02:25,460] Trial 0 finished with value: 0.7016574585635359 and parameters: {'n_estimators': 608, 'max_depth': 2, 'learning_rate': 0.1540596442758995, 'num_leaves': 101, 'subsample': 0.7700929605373403, 'colsample_bytree': 0.9151473393300822}. Best is trial 0 with value: 0.7016574585635359.
[I 2026-02-23 19:02:26,944] Trial 1 finished with value: 0.6763005780346821 and parameters: {'n_estimators': 937, 'max_depth': 13, 'learning_rate': 0.26214633806506055, 'num_leaves': 127, 'subsample': 0.7439978417121977, 'colsample_bytree': 0.698589381773225}. Best is trial 0 with value: 0.7016574585635359.
[I 2026-02-23 19:02:28,851] Trial 2 finished with value: 0.696165191740413 and parameters: {'n_estimators': 964, 'max_depth': 16, 'learning_rate': 0.12325162301299777, 'num_leaves': 185, 'subsample': 0.8150676349905709, 'colsample_bytree': 0.6788466615374116}. Best is trial 0 with value: 0.7016574585635359.
[I 2026-02-23 19:02:30,146] Trial 3 finished with value: 0.725146198830409

Best VAL F1 (LGBM): 0.7251

================ MODEL COMPARISON (VAL) ================
RF     | VAL F1 = 0.7130
XGB    | VAL F1 = 0.7317
LGBM   | VAL F1 = 0.7251

 WINNER: XGB

 Retraining best model on TRAIN + VAL...
Class distribution (train+val): Counter({np.int64(0): 2944, np.int64(1): 1276})
New scale_pos_weight: 2.307

Model saved as medium_1.pkl

 FINAL TEST F1 (XGB): 0.7306

=== Classification Report (TEST SET) ===
              precision    recall  f1-score   support

 Less potent     0.8900    0.8670    0.8784       737
      Potent     0.7101    0.7524    0.7306       319

    accuracy                         0.8324      1056
   macro avg     0.8000    0.8097    0.8045      1056
weighted avg     0.8356    0.8324    0.8337      1056



## Model 2

In [13]:
# =============================================================================
# CELL 1: Load data → Remove Grey Zone → Split
# =============================================================================

import pandas as pd
from sklearn.model_selection import train_test_split

# -------------------------------------------------------------------------
# 1. LOAD FEATURE FILE
# -------------------------------------------------------------------------

df = pd.read_parquet("wildtype egfr_features.parquet")
print(f"Total samples loaded: {len(df)}")

assert "y" in df.columns, "Column y not found!"
assert "pIC50 (M)" in df.columns, "Column pIC50 (M) not found!"

df.columns = [
    col.replace("ECFP6_", "ECFP_") if col.startswith("ECFP6_") else col
    for col in df.columns
]

df.columns = [
    col.replace("Pharm2D_", "Pharm_") if col.startswith("Pharm2D_") else col
    for col in df.columns
]

# -------------------------------------------------------------------------
# 2. REMOVE GREY ZONE (pIC50 in [7.2, 7.8])
# -------------------------------------------------------------------------

before_gz = len(df)

df = df[~df["pIC50 (M)"].between(7.2, 7.8)]

after_gz = len(df)

print(f"Removed grey zone samples: {before_gz - after_gz}")
print(f"Remaining samples: {after_gz}")

# -------------------------------------------------------------------------
# 3. SELECT FINGERPRINT COLUMNS
# -------------------------------------------------------------------------

ecfp_cols = [c for c in df.columns if c.startswith("ECFP_")]
pharm_cols = [c for c in df.columns if c.startswith("Pharm_")]

print(f"ECFP features   : {len(ecfp_cols)}")
print(f"Pharm2D features: {len(pharm_cols)}")

feature_cols = ecfp_cols + pharm_cols

X = df[feature_cols].copy()
y = df["y"].copy()

print(f"Total fingerprint features used: {X.shape[1]}")

# -------------------------------------------------------------------------
# 4. TRAIN / VAL / TEST SPLIT (70 / 10 / 20)
# -------------------------------------------------------------------------

X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp,
    test_size=0.125,  # 0.125 × 0.8 = 0.10
    random_state=42,
    stratify=y_tmp
)

print(f"Train size      : {X_train.shape}")
print(f"Validation size : {X_val.shape}")
print(f"Test size       : {X_test.shape}")

print("Cell 1 finished: Grey zone removed and data ready.")

Total samples loaded: 5276
Removed grey zone samples: 775
Remaining samples: 4501
ECFP features   : 2048
Pharm2D features: 39972
Total fingerprint features used: 42020
Train size      : (3150, 42020)
Validation size : (450, 42020)
Test size       : (901, 42020)
Cell 1 finished: Grey zone removed and data ready.


In [14]:
# =============================================================================
# CELL 2: FEATURE SELECTION: Variance → Mutual Information
# =============================================================================

from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import mutual_info_classif
import numpy as np

# -------------------------------------------------------------------------
# 1. VARIANCE THRESHOLD (binary fingerprints)
# -------------------------------------------------------------------------

print("Before VarianceThreshold:", X_train.shape)

vt = VarianceThreshold(threshold=0.05)  # remove near-constant bits
X_train_vt = vt.fit_transform(X_train)
X_val_vt   = vt.transform(X_val)
X_test_vt  = vt.transform(X_test)

print("After VarianceThreshold:", X_train_vt.shape)

# -------------------------------------------------------------------------
# 2. MUTUAL INFORMATION
# -------------------------------------------------------------------------

mi = mutual_info_classif(
    X_train_vt,
    y_train,
    random_state=42
)

# Keep top 50%
mi_threshold = np.percentile(mi, 50)
selected_idx = np.where(mi >= mi_threshold)[0]

X_train_final = X_train_vt[:, selected_idx]
X_val_final   = X_val_vt[:, selected_idx]
X_test_final  = X_test_vt[:, selected_idx]

print("After Mutual Information:", X_train_final.shape)

print("Cell 2 finished: feature selection completed")

Before VarianceThreshold: (3150, 42020)
After VarianceThreshold: (3150, 3035)
After Mutual Information: (3150, 1518)
Cell 2 finished: feature selection completed


In [ ]:
# =============================================================================
# CELL 3: OPTUNA → MODEL COMPARISON (RF / XGB / LGBM)
# =============================================================================

import optuna
import warnings
warnings.filterwarnings("ignore")

from collections import Counter
from sklearn.metrics import f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier

import xgboost as xgb
import lightgbm as lgb
import numpy as np

import random
import numpy as np
import os

random.seed(42)
np.random.seed(42)
os.environ["PYTHONHASHSEED"] = "42"

import joblib

# -------------------------------------------------------------------------
# 0. HANDLE IMBALANCE (TRAIN SET ONLY)
# -------------------------------------------------------------------------
counter = Counter(y_train)
n_neg = counter[0]
n_pos = counter[1]
scale_pos_weight = n_neg / n_pos

print(f"Class distribution (train): {counter}")
print(f"scale_pos_weight (XGB/LGBM): {scale_pos_weight:.3f}")

# -------------------------------------------------------------------------
# 1. OBJECTIVE FUNCTION (TUNING ON VALIDATION SET)
# -------------------------------------------------------------------------
def objective(trial, model_name):

    if model_name == "RF":
        model = RandomForestClassifier(
            n_estimators=trial.suggest_int("n_estimators", 200, 800),
            max_depth=trial.suggest_int("max_depth", 5, 30),
            min_samples_split=trial.suggest_int("min_samples_split", 2, 10),
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 5),
            max_features=trial.suggest_categorical("max_features", ["sqrt", "log2"]),
            class_weight="balanced",
            n_jobs=1,
            random_state=42
        )
        model.fit(X_train_final, y_train)

    elif model_name == "XGB":
        model = xgb.XGBClassifier(
            n_estimators=trial.suggest_int("n_estimators", 200, 500),
            max_depth=trial.suggest_int("max_depth", 4, 8),
            learning_rate=trial.suggest_float("learning_rate", 0.03, 0.15, log=True),
            subsample=trial.suggest_float("subsample", 0.7, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 0.8),
            tree_method="hist",
            eval_metric="logloss",
            scale_pos_weight=scale_pos_weight,
            n_jobs=1,
            random_state=42,
            seed=42
        )
        model.fit(X_train_final, y_train)

    elif model_name == "LGBM":
        model = lgb.LGBMClassifier(
            n_estimators=trial.suggest_int("n_estimators", 300, 1000),
            max_depth=trial.suggest_int("max_depth", -1, 20),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            num_leaves=trial.suggest_int("num_leaves", 31, 256),
            subsample=trial.suggest_float("subsample", 0.6, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
            scale_pos_weight=scale_pos_weight, # Using weight instead of 'balanced' for consistency
            n_jobs=1,
            random_state=42,
            feature_fraction_seed=42,
            bagging_seed=42,
            data_random_seed=42,
            verbose=-1
        )
        model.fit(
            X_train_final, y_train,
            eval_set=[(X_val_final, y_val)],
            eval_metric="f1",
            callbacks=[lgb.early_stopping(50, verbose=False)]
        )
    else:
        raise ValueError("Unknown model")

    # VALIDATION SCORE ONLY
    y_val_pred = model.predict(X_val_final)
    return f1_score(y_val, y_val_pred)

# -------------------------------------------------------------------------
# 2. RUN OPTUNA
# -------------------------------------------------------------------------
models = ["RF", "XGB", "LGBM"]
results = {}

for model_name in models:
    print(f"\n Optimizing {model_name}...")
    study = optuna.create_study(direction="maximize")
    study.optimize(lambda trial: objective(trial, model_name), n_trials=30)

    results[model_name] = {
        "best_val_f1": study.best_value,
        "best_params": study.best_params
    }
    print(f"Best VAL F1 ({model_name}): {study.best_value:.4f}")

# -------------------------------------------------------------------------
# 3. MODEL COMPARISON (BASED ON VAL SCORE)
# -------------------------------------------------------------------------
best_model_name = max(results, key=lambda k: results[k]["best_val_f1"])
best_params = results[best_model_name]["best_params"]

print("\n================ MODEL COMPARISON (VAL) ================")
for m, res in results.items():
    print(f"{m:6s} | VAL F1 = {res['best_val_f1']:.4f}")

print(f"\n WINNER: {best_model_name}")

# -------------------------------------------------------------------------
# 4. FINAL RETRAIN (TRAIN + VAL) AND TEST EVALUATION
# -------------------------------------------------------------------------
print("\n Retraining best model on TRAIN + VAL...")

X_trainval = np.vstack([X_train_final, X_val_final])
y_trainval = np.hstack([y_train, y_val])

# Recompute scale_pos_weight on full training data (train + val)
from collections import Counter

counter_trainval = Counter(y_trainval)
n_neg_tv = counter_trainval[0]
n_pos_tv = counter_trainval[1]
scale_pos_weight_tv = n_neg_tv / n_pos_tv

print(f"Class distribution (train+val): {counter_trainval}")
print(f"Recomputed scale_pos_weight: {scale_pos_weight_tv:.3f}")

# Build final model
if best_model_name == "RF":
    final_model = RandomForestClassifier(
        **best_params,
        class_weight="balanced",
        n_jobs=-1,
        random_state=42
    )

elif best_model_name == "XGB":
    final_model = xgb.XGBClassifier(
        **best_params,
        scale_pos_weight=scale_pos_weight_tv,
        n_jobs=-1,
        random_state=42,
        seed=42
    )

elif best_model_name == "LGBM":
    final_model = lgb.LGBMClassifier(
        **best_params,
        scale_pos_weight=scale_pos_weight_tv,
        n_jobs=-1,
        random_state=42,
        verbose=-1
    )

# Fit on full training data
final_model.fit(X_trainval, y_trainval)

# Save model
model_package = {
    "model": final_model,
    "model_name": best_model_name,
    "best_params": best_params,
    "variance_selector": vt,
    "selected_idx": selected_idx
}

joblib.dump(model_package, "model_2.pkl")
print("\nModel saved as model_2.pkl")

# ---------------- TEST EVALUATION ----------------
y_test_pred = final_model.predict(X_test_final)
test_f1 = f1_score(y_test, y_test_pred)

print(f"\n FINAL TEST F1 ({best_model_name}): {test_f1:.4f}")
print("\n=== Classification Report (TEST SET) ===")
print(classification_report(y_test, y_test_pred,
                            target_names=["Less potent", "Potent"],
                            digits=4))

[I 2026-02-23 17:49:57,083] A new study created in memory with name: no-name-7060c21a-df5d-45fc-a366-a432342a94a8


Class distribution (train): Counter({0: 2299, 1: 851})
scale_pos_weight (XGB/LGBM): 2.702

 Optimizing RF...


[I 2026-02-23 17:50:01,346] Trial 0 finished with value: 0.7705627705627706 and parameters: {'n_estimators': 253, 'max_depth': 19, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.7705627705627706.
[I 2026-02-23 17:50:03,525] Trial 1 finished with value: 0.7168458781362007 and parameters: {'n_estimators': 320, 'max_depth': 10, 'min_samples_split': 3, 'min_samples_leaf': 5, 'max_features': 'log2'}. Best is trial 0 with value: 0.7705627705627706.
[I 2026-02-23 17:50:11,638] Trial 2 finished with value: 0.7692307692307693 and parameters: {'n_estimators': 670, 'max_depth': 22, 'min_samples_split': 3, 'min_samples_leaf': 1, 'max_features': 'log2'}. Best is trial 0 with value: 0.7705627705627706.
[I 2026-02-23 17:50:14,173] Trial 3 finished with value: 0.7132867132867133 and parameters: {'n_estimators': 643, 'max_depth': 8, 'min_samples_split': 3, 'min_samples_leaf': 1, 'max_features': 'log2'}. Best is trial 0 with value: 0.770562770562770

Best VAL F1 (RF): 0.8017

 Optimizing XGB...


[I 2026-02-23 17:52:46,518] Trial 0 finished with value: 0.8044280442804428 and parameters: {'n_estimators': 332, 'max_depth': 4, 'learning_rate': 0.03312702446459501, 'subsample': 0.7823479750591961, 'colsample_bytree': 0.7541991342452428}. Best is trial 0 with value: 0.8044280442804428.
[I 2026-02-23 17:52:52,931] Trial 1 finished with value: 0.788135593220339 and parameters: {'n_estimators': 332, 'max_depth': 7, 'learning_rate': 0.08192406213706489, 'subsample': 0.8953388765757209, 'colsample_bytree': 0.559724480214392}. Best is trial 0 with value: 0.8044280442804428.
[I 2026-02-23 17:53:00,035] Trial 2 finished with value: 0.8170212765957446 and parameters: {'n_estimators': 311, 'max_depth': 8, 'learning_rate': 0.04900321210122953, 'subsample': 0.977145042192587, 'colsample_bytree': 0.676354111434552}. Best is trial 2 with value: 0.8170212765957446.
[I 2026-02-23 17:53:05,895] Trial 3 finished with value: 0.8286852589641435 and parameters: {'n_estimators': 300, 'max_depth': 6, 'lea

Best VAL F1 (XGB): 0.8412

 Optimizing LGBM...


[I 2026-02-23 17:55:42,048] Trial 0 finished with value: 0.8067226890756303 and parameters: {'n_estimators': 485, 'max_depth': 14, 'learning_rate': 0.18689508206795777, 'num_leaves': 59, 'subsample': 0.718975675843325, 'colsample_bytree': 0.9330009896047602}. Best is trial 0 with value: 0.8067226890756303.
[I 2026-02-23 17:55:42,948] Trial 1 finished with value: 0.7491166077738516 and parameters: {'n_estimators': 388, 'max_depth': 2, 'learning_rate': 0.041416022960291614, 'num_leaves': 89, 'subsample': 0.605703743148221, 'colsample_bytree': 0.9755771630673598}. Best is trial 0 with value: 0.8067226890756303.
[I 2026-02-23 17:55:47,558] Trial 2 finished with value: 0.8185654008438819 and parameters: {'n_estimators': 512, 'max_depth': 15, 'learning_rate': 0.016761361191295492, 'num_leaves': 113, 'subsample': 0.9585459462553786, 'colsample_bytree': 0.7791382322818724}. Best is trial 2 with value: 0.8185654008438819.
[I 2026-02-23 17:55:49,537] Trial 3 finished with value: 0.81481481481481

Best VAL F1 (LGBM): 0.8299

================ MODEL COMPARISON (VAL) ================
RF     | VAL F1 = 0.8017
XGB    | VAL F1 = 0.8412
LGBM   | VAL F1 = 0.8299

 WINNER: XGB

 Retraining best model on TRAIN + VAL...
Class distribution (train+val): Counter({np.int64(0): 2628, np.int64(1): 972})
Recomputed scale_pos_weight: 2.704

Model saved as medium_2.pkl

 FINAL TEST F1 (XGB): 0.7702

=== Classification Report (TEST SET) ===
              precision    recall  f1-score   support

 Less potent     0.9198    0.9058    0.9127       658
      Potent     0.7549    0.7860    0.7702       243

    accuracy                         0.8735       901
   macro avg     0.8373    0.8459    0.8414       901
weighted avg     0.8753    0.8735    0.8743       901



## Model 3

In [1]:
# =============================================================================
# CELL 1: Load data → Remove Grey Zone → Split
# =============================================================================

import pandas as pd
from sklearn.model_selection import train_test_split

# -------------------------------------------------------------------------
# 1. LOAD FEATURE FILE
# -------------------------------------------------------------------------

df = pd.read_parquet("wildtype egfr_features.parquet")
print(f"Total samples loaded: {len(df)}")

assert "y" in df.columns, "Column y not found!"
assert "pIC50 (M)" in df.columns, "Column pIC50 (M) not found!"

df.columns = [
    col.replace("ECFP6_", "ECFP_") if col.startswith("ECFP6_") else col
    for col in df.columns
]

df.columns = [
    col.replace("Pharm2D_", "Pharm_") if col.startswith("Pharm2D_") else col
    for col in df.columns
]

# -------------------------------------------------------------------------
# 2. REMOVE GREY ZONE (pIC50 in [7.2, 7.8])
# -------------------------------------------------------------------------

before_gz = len(df)

df = df[~df["pIC50 (M)"].between(7.2, 7.8)]

after_gz = len(df)

print(f"Removed grey zone samples: {before_gz - after_gz}")
print(f"Remaining samples: {after_gz}")

# -------------------------------------------------------------------------
# 3. SELECT FINGERPRINT COLUMNS
# -------------------------------------------------------------------------

ecfp_cols = [c for c in df.columns if c.startswith("ECFP_")]
pharm_cols = [c for c in df.columns if c.startswith("Pharm_")]

print(f"ECFP features   : {len(ecfp_cols)}")
print(f"Pharm2D features: {len(pharm_cols)}")

feature_cols = ecfp_cols + pharm_cols

X = df[feature_cols].copy()
y = df["y"].copy()

print(f"Total fingerprint features used: {X.shape[1]}")

# -------------------------------------------------------------------------
# 4. TRAIN / TEST SPLIT (80 / 20)
# -------------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train size : {X_train.shape}")
print(f"Test size  : {X_test.shape}")

print("Cell 1 finished: grey zone removed and data ready")

Total samples loaded: 5276
Removed grey zone samples: 775
Remaining samples: 4501
ECFP features   : 2048
Pharm2D features: 39972
Total fingerprint features used: 42020
Train size : (3600, 42020)
Test size  : (901, 42020)
Cell 1 finished: grey zone removed and data ready


In [ ]:
# =============================================================================
# CELL 2 (Preliminary): OPTUNA + 3-FOLD CV + CLASS WEIGHT + FEATURE SELECTION INSIDE FOLD + FINAL MODEL SAVING + TEST EVALUATION
# =============================================================================

import optuna
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import random
import os

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold, SelectPercentile, mutual_info_classif
from sklearn.pipeline import Pipeline
from collections import Counter

import xgboost as xgb
import lightgbm as lgb
import joblib

# -------------------------------------------------------------------------
# 0. SEED EVERYTHING
# -------------------------------------------------------------------------
random.seed(42)
np.random.seed(42)
os.environ["PYTHONHASHSEED"] = "42"

# -------------------------------------------------------------------------
# 1. GLOBAL CLASS IMBALANCE (FOR FINAL MODEL)
# -------------------------------------------------------------------------
counter = Counter(y_train)
n_neg = counter[0]
n_pos = counter[1]
scale_pos_weight = n_neg / n_pos

print(f"Class distribution (train): {counter}")
print(f"Global scale_pos_weight: {scale_pos_weight:.3f}")

# -------------------------------------------------------------------------
# 2. STRATIFIED 3-FOLD
# -------------------------------------------------------------------------
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# -------------------------------------------------------------------------
# 3. OBJECTIVE FUNCTION
# -------------------------------------------------------------------------
def objective(trial, model_name):

    f1_scores = []

    for train_idx, val_idx in cv.split(X_train, y_train):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val_fold = y_train.iloc[val_idx]

        fold_counter = Counter(y_tr)
        fold_scale_pos_weight = fold_counter[0] / fold_counter[1]

        # ----------------- MODEL DEFINITION -----------------
        if model_name == "RF":
            clf = RandomForestClassifier(
                n_estimators=trial.suggest_int("n_estimators", 200, 800),
                max_depth=trial.suggest_int("max_depth", 5, 30),
                min_samples_split=trial.suggest_int("min_samples_split", 2, 10),
                min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 5),
                max_features=trial.suggest_categorical("max_features", ["sqrt", "log2"]),
                class_weight="balanced",
                n_jobs=1,
                random_state=42
            )

        elif model_name == "XGB":
            clf = xgb.XGBClassifier(
                n_estimators=trial.suggest_int("n_estimators", 200, 500),
                max_depth=trial.suggest_int("max_depth", 4, 8),
                learning_rate=trial.suggest_float("learning_rate", 0.03, 0.15, log=True),
                subsample=trial.suggest_float("subsample", 0.7, 1.0),
                colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 0.8),
                scale_pos_weight=fold_scale_pos_weight,
                tree_method="hist",
                eval_metric="logloss",
                n_jobs=1,
                random_state=42
            )

        elif model_name == "LGBM":
            clf = lgb.LGBMClassifier(
                n_estimators=trial.suggest_int("n_estimators", 300, 1000),
                max_depth=trial.suggest_int("max_depth", -1, 20),
                learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                num_leaves=trial.suggest_int("num_leaves", 31, 256),
                subsample=trial.suggest_float("subsample", 0.6, 1.0),
                colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
                scale_pos_weight=fold_scale_pos_weight,
                n_jobs=1,
                random_state=42,
                verbose=-1
            )

        # ----------------- PIPELINE -----------------
        pipeline = Pipeline([
            ("vt", VarianceThreshold(threshold=0.05)),
            ("mi", SelectPercentile(mutual_info_classif, percentile=50)),
            ("clf", clf)
        ])

        pipeline.fit(X_tr, y_tr)
        y_val_pred = pipeline.predict(X_val)

        f1_scores.append(f1_score(y_val_fold, y_val_pred))

    return np.mean(f1_scores)

# -------------------------------------------------------------------------
# 4. RUN OPTUNA
# -------------------------------------------------------------------------
models = ["RF", "XGB", "LGBM"]
results = {}

for model_name in models:

    print(f"\nOptimizing {model_name}...")

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42)
    )

    study.optimize(lambda trial: objective(trial, model_name), n_trials=10)

    results[model_name] = {
        "best_cv_f1": study.best_value,
        "best_params": study.best_params
    }

    print(f"Best CV F1 ({model_name}): {study.best_value:.4f}")

# -------------------------------------------------------------------------
# 5. MODEL COMPARISON
# -------------------------------------------------------------------------
print("\n================ MODEL COMPARISON (3-FOLD CV) ================")

for m, res in results.items():
    print(f"{m:6s} | Mean CV F1 = {res['best_cv_f1']:.4f}")

best_model_name = max(results, key=lambda k: results[k]["best_cv_f1"])
best_params = results[best_model_name]["best_params"]

print(f"\n WINNER: {best_model_name}")
print(f"Best CV F1: {results[best_model_name]['best_cv_f1']:.4f}")

# -------------------------------------------------------------------------
# 6. FINAL TRAIN ON FULL TRAIN SET
# -------------------------------------------------------------------------
print("\nRetraining best model on full TRAIN set...")

if best_model_name == "RF":
    final_clf = RandomForestClassifier(
        **best_params,
        class_weight="balanced",
        n_jobs=-1,
        random_state=42
    )

elif best_model_name == "XGB":
    final_clf = xgb.XGBClassifier(
        **best_params,
        scale_pos_weight=scale_pos_weight,
        n_jobs=-1,
        random_state=42
    )

elif best_model_name == "LGBM":
    final_clf = lgb.LGBMClassifier(
        **best_params,
        scale_pos_weight=scale_pos_weight,
        n_jobs=-1,
        random_state=42,
        verbose=-1
    )

final_model = Pipeline([
    ("vt", VarianceThreshold(threshold=0.05)),
    ("mi", SelectPercentile(mutual_info_classif, percentile=50)),
    ("clf", final_clf)
])

final_model.fit(X_train, y_train)

# -------------------------------------------------------------------------
# 7. FINAL TEST EVALUATION
# -------------------------------------------------------------------------
print("\n================ FINAL TEST EVALUATION ================")

y_test_pred = final_model.predict(X_test)
test_f1 = f1_score(y_test, y_test_pred)

print(f"\nFINAL TEST F1 ({best_model_name}): {test_f1:.4f}")

print("\n=== Classification Report (TEST SET) ===")
print(classification_report(
    y_test,
    y_test_pred,
    target_names=["Less potent", "Potent"],
    digits=4
))

[I 2026-02-26 18:03:56,136] A new study created in memory with name: no-name-63d4d6aa-f340-49ff-bfe6-57f01d00dfa8


Class distribution (train): Counter({0: 2628, 1: 972})
Global scale_pos_weight: 2.704

Optimizing RF...


[I 2026-02-26 18:09:28,295] Trial 0 finished with value: 0.7608682234963334 and parameters: {'n_estimators': 425, 'max_depth': 29, 'min_samples_split': 8, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.7608682234963334.
[I 2026-02-26 18:14:22,656] Trial 1 finished with value: 0.7302775748024558 and parameters: {'n_estimators': 234, 'max_depth': 27, 'min_samples_split': 7, 'min_samples_leaf': 4, 'max_features': 'log2'}. Best is trial 0 with value: 0.7608682234963334.
[I 2026-02-26 18:19:42,573] Trial 2 finished with value: 0.7270604234175521 and parameters: {'n_estimators': 700, 'max_depth': 10, 'min_samples_split': 3, 'min_samples_leaf': 1, 'max_features': 'log2'}. Best is trial 0 with value: 0.7608682234963334.
[I 2026-02-26 18:25:11,059] Trial 3 finished with value: 0.7312351479302871 and parameters: {'n_estimators': 459, 'max_depth': 12, 'min_samples_split': 7, 'min_samples_leaf': 1, 'max_features': 'log2'}. Best is trial 0 with value: 0.76086822349633

Best CV F1 (RF): 0.7609

Optimizing XGB...


[I 2026-02-26 19:05:10,837] Trial 0 finished with value: 0.776126911004169 and parameters: {'n_estimators': 312, 'max_depth': 8, 'learning_rate': 0.09744578609310868, 'subsample': 0.8795975452591109, 'colsample_bytree': 0.5468055921327309}. Best is trial 0 with value: 0.776126911004169.
[I 2026-02-26 19:10:37,324] Trial 1 finished with value: 0.7648991534163129 and parameters: {'n_estimators': 246, 'max_depth': 4, 'learning_rate': 0.12093510864110771, 'subsample': 0.8803345035229626, 'colsample_bytree': 0.7124217733388136}. Best is trial 0 with value: 0.776126911004169.
[I 2026-02-26 19:13:38,424] Trial 2 finished with value: 0.772563344333108 and parameters: {'n_estimators': 206, 'max_depth': 8, 'learning_rate': 0.11454435497690606, 'subsample': 0.7637017332034828, 'colsample_bytree': 0.5545474901621302}. Best is trial 0 with value: 0.776126911004169.
[I 2026-02-26 19:16:00,179] Trial 3 finished with value: 0.7737890336857633 and parameters: {'n_estimators': 255, 'max_depth': 5, 'lear

Best CV F1 (XGB): 0.7804

Optimizing LGBM...


[I 2026-02-26 19:32:31,660] Trial 0 finished with value: 0.7618144092420619 and parameters: {'n_estimators': 562, 'max_depth': 19, 'learning_rate': 0.1205712628744377, 'num_leaves': 166, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481}. Best is trial 0 with value: 0.7618144092420619.
[I 2026-02-26 19:34:49,757] Trial 1 finished with value: 0.7687110845888033 and parameters: {'n_estimators': 340, 'max_depth': 18, 'learning_rate': 0.07725378389307355, 'num_leaves': 191, 'subsample': 0.608233797718321, 'colsample_bytree': 0.9879639408647978}. Best is trial 1 with value: 0.7687110845888033.
[I 2026-02-26 19:37:04,340] Trial 2 finished with value: 0.7535805842148479 and parameters: {'n_estimators': 883, 'max_depth': 3, 'learning_rate': 0.01855998084649059, 'num_leaves': 72, 'subsample': 0.7216968971838151, 'colsample_bytree': 0.8099025726528951}. Best is trial 1 with value: 0.7687110845888033.
[I 2026-02-26 19:39:17,011] Trial 3 finished with value: 0.7719741096355609

Best CV F1 (LGBM): 0.7720

================ MODEL COMPARISON (5-FOLD CV) ================
RF     | Mean CV F1 = 0.7609
XGB    | Mean CV F1 = 0.7804
LGBM   | Mean CV F1 = 0.7720

 WINNER: XGB
Best CV F1: 0.7804

Retraining best model on full TRAIN set...

================ FINAL TEST EVALUATION ================

FINAL TEST F1 (XGB): 0.7812

=== Classification Report (TEST SET) ===
              precision    recall  f1-score   support

 Less potent     0.9320    0.8951    0.9132       658
      Potent     0.7435    0.8230    0.7812       243

    accuracy                         0.8757       901
   macro avg     0.8377    0.8591    0.8472       901
weighted avg     0.8811    0.8757    0.8776       901



In [ ]:
# =============================================================================
# CELL 3: FINAL XGB FINE-TUNING (5-FOLD CV + 15 TRIALS)
# =============================================================================

import optuna
import numpy as np
import random
import os
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.feature_selection import VarianceThreshold, SelectPercentile, mutual_info_classif
from sklearn.pipeline import Pipeline
from collections import Counter
import xgboost as xgb
import joblib

# -------------------------------------------------------------------------
# 0. SEED EVERYTHING
# -------------------------------------------------------------------------
random.seed(42)
np.random.seed(42)
os.environ["PYTHONHASHSEED"] = "42"

# -------------------------------------------------------------------------
# 1. GLOBAL CLASS IMBALANCE (FOR FINAL MODEL)
# -------------------------------------------------------------------------
counter = Counter(y_train)
n_neg = counter[0]
n_pos = counter[1]
scale_pos_weight = n_neg / n_pos

print(f"Class distribution (train): {counter}")
print(f"Global scale_pos_weight: {scale_pos_weight:.3f}")

# -------------------------------------------------------------------------
# 2. STRATIFIED 5-FOLD
# -------------------------------------------------------------------------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# -------------------------------------------------------------------------
# 3. OBJECTIVE FUNCTION (XGB ONLY)
# -------------------------------------------------------------------------
def objective(trial):

    f1_scores = []

    for train_idx, val_idx in cv.split(X_train, y_train):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val_fold = y_train.iloc[val_idx]

        fold_counter = Counter(y_tr)
        fold_scale_pos_weight = fold_counter[0] / fold_counter[1]

        clf = xgb.XGBClassifier(
            n_estimators=trial.suggest_int("n_estimators", 200, 600),
            max_depth=trial.suggest_int("max_depth", 4, 8),
            learning_rate=trial.suggest_float("learning_rate", 0.02, 0.15, log=True),
            subsample=trial.suggest_float("subsample", 0.7, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 0.9),
            scale_pos_weight=fold_scale_pos_weight,
            tree_method="hist",
            eval_metric="logloss",
            n_jobs=-1,
            random_state=42
        )

        pipeline = Pipeline([
            ("vt", VarianceThreshold(threshold=0.05)),
            ("mi", SelectPercentile(mutual_info_classif, percentile=50)),
            ("clf", clf)
        ])

        pipeline.fit(X_tr, y_tr)
        y_val_pred = pipeline.predict(X_val)

        f1_scores.append(f1_score(y_val_fold, y_val_pred))

    return np.mean(f1_scores)

# -------------------------------------------------------------------------
# 4. RUN OPTUNA (15 TRIALS)
# -------------------------------------------------------------------------
print("\nOptimizing XGB (Final Phase)...")

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42)
)

study.optimize(objective, n_trials=15)

best_params = study.best_params
print(f"\nBest CV F1 (XGB - 5 fold): {study.best_value:.4f}")

# -------------------------------------------------------------------------
# 5. FINAL TRAIN ON FULL TRAIN SET
# -------------------------------------------------------------------------
print("\nRetraining final XGB model on full TRAIN set...")

final_clf = xgb.XGBClassifier(
    **best_params,
    scale_pos_weight=scale_pos_weight,
    tree_method="hist",
    eval_metric="logloss",
    n_jobs=-1,
    random_state=42
)

final_model = Pipeline([
    ("vt", VarianceThreshold(threshold=0.05)),
    ("mi", SelectPercentile(mutual_info_classif, percentile=50)),
    ("clf", final_clf)
])

final_model.fit(X_train, y_train)

# -------------------------------------------------------------------------
# 6. FINAL TEST EVALUATION
# -------------------------------------------------------------------------
print("\n================ FINAL TEST EVALUATION ================")

y_test_pred = final_model.predict(X_test)
test_f1 = f1_score(y_test, y_test_pred)

print(f"\nFINAL TEST F1 (XGB): {test_f1:.4f}")

print("\n=== Classification Report (TEST SET) ===")
print(classification_report(
    y_test,
    y_test_pred,
    target_names=["Less potent", "Potent"],
    digits=4
))

# -------------------------------------------------------------------------
# 7. SAVE FINAL MODEL
# -------------------------------------------------------------------------
joblib.dump(final_model, "model_3.pkl")
print("\nModel saved as model_3.pkl")

[I 2026-02-26 20:32:42,587] A new study created in memory with name: no-name-79ada13b-ab0f-4bd7-9a15-98acab9fac05


Class distribution (train): Counter({0: 2628, 1: 972})
Global scale_pos_weight: 2.704

Optimizing XGB (Final Phase)...


[I 2026-02-26 20:38:51,636] Trial 0 finished with value: 0.7825153763500763 and parameters: {'n_estimators': 350, 'max_depth': 8, 'learning_rate': 0.08741169448864673, 'subsample': 0.8795975452591109, 'colsample_bytree': 0.5624074561769746}. Best is trial 0 with value: 0.7825153763500763.
[I 2026-02-26 20:46:30,150] Trial 1 finished with value: 0.7886160841102756 and parameters: {'n_estimators': 262, 'max_depth': 4, 'learning_rate': 0.11454791487656053, 'subsample': 0.8803345035229626, 'colsample_bytree': 0.7832290311184182}. Best is trial 1 with value: 0.7886160841102756.
[I 2026-02-26 20:54:17,008] Trial 2 finished with value: 0.7838196616068958 and parameters: {'n_estimators': 208, 'max_depth': 8, 'learning_rate': 0.10702082748695269, 'subsample': 0.7637017332034828, 'colsample_bytree': 0.5727299868828403}. Best is trial 1 with value: 0.7886160841102756.
[I 2026-02-26 21:02:04,739] Trial 3 finished with value: 0.7820556042436795 and parameters: {'n_estimators': 273, 'max_depth': 5, 


Best CV F1 (XGB - 5 fold): 0.7925

Retraining final XGB model on full TRAIN set...

================ FINAL TEST EVALUATION ================

FINAL TEST F1 (XGB): 0.7847

=== Classification Report (TEST SET) ===
              precision    recall  f1-score   support

 Less potent     0.9258    0.9103    0.9180       658
      Potent     0.7677    0.8025    0.7847       243

    accuracy                         0.8812       901
   macro avg     0.8468    0.8564    0.8514       901
weighted avg     0.8832    0.8812    0.8821       901


Model saved as model_3.pkl


# Load model

## Descriptors Comparison

In [31]:
# =============================================================================
# REPRODUCE FINAL RANKING FROM SAVED MODELS (FULLY CORRECT VERSION)
# =============================================================================

import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

# -------------------------------------------------------------------------
# SETTINGS
# -------------------------------------------------------------------------
PARQUET_FILE = "wildtype egfr_features.parquet"
MODEL_FOLDER = "saved_models"
SEED = 42

fingerprint_names = [
    "ECFP",
    "Pharm2D",
    "RDKit",
    "ECFP+Pharm2D",
    "ECFP+RDKit",
    "Pharm2D+RDKit",
    "ALL"
]

# -------------------------------------------------------------------------
# 1. LOAD FEATURE MATRIX
# -------------------------------------------------------------------------
df = pd.read_parquet(PARQUET_FILE)

y = df["y"].values
X_features = df.drop(columns=["SMILES", "y"])

# Reproduce original split (must match training)
X_train_base, X_test_base, y_train, y_test = train_test_split(
    X_features,
    y,
    test_size=0.2,
    stratify=y,
    random_state=SEED
)

train_idx = X_train_base.index
test_idx  = X_test_base.index

# Split feature types
X_ecfp  = X_features.filter(regex="^ECFP_")
X_pharm = X_features.filter(regex="^Pharm_")
X_rdkit = X_features.filter(regex="^RDKit_")

results = {}

# -------------------------------------------------------------------------
# 2. LOOP THROUGH ALL FINGERPRINT SETS
# -------------------------------------------------------------------------
for name in fingerprint_names:

    model_dict = joblib.load(f"{MODEL_FOLDER}/{name}.pkl")

    best_model_name = model_dict["best_model_name"]
    selected_idx    = model_dict["selected_idx"]
    vt_bin          = model_dict["vt_bin"]
    vt_cont         = model_dict["vt_cont"]
    imputer         = model_dict["imputer"]
    scaler          = model_dict["scaler"]

    rf_model  = model_dict["rf"]
    xgb_model = model_dict["xgb"]
    lgb_model = model_dict["lgbm"]

    if best_model_name == "RF":
        model = rf_model
    elif best_model_name == "XGB":
        model = xgb_model
    else:
        model = lgb_model

    # -----------------------------------------------------------------
    # Rebuild fingerprint combination
    # -----------------------------------------------------------------
    if name == "ECFP":
        X_bin_all, X_cont_all = X_ecfp, None

    elif name == "Pharm2D":
        X_bin_all, X_cont_all = X_pharm, None

    elif name == "RDKit":
        X_bin_all, X_cont_all = None, X_rdkit

    elif name == "ECFP+Pharm2D":
        X_bin_all = pd.concat([X_ecfp, X_pharm], axis=1)
        X_cont_all = None

    elif name == "ECFP+RDKit":
        X_bin_all, X_cont_all = X_ecfp, X_rdkit

    elif name == "Pharm2D+RDKit":
        X_bin_all, X_cont_all = X_pharm, X_rdkit

    elif name == "ALL":
        X_bin_all = pd.concat([X_ecfp, X_pharm], axis=1)
        X_cont_all = X_rdkit

    # Select test set only
    if X_bin_all is not None:
        X_bin = X_bin_all.loc[test_idx].copy()
    else:
        X_bin = None

    if X_cont_all is not None:
        X_cont = X_cont_all.loc[test_idx].copy()
    else:
        X_cont = None

    # -----------------------------------------------------------------
    # Apply preprocessing EXACTLY as training
    # -----------------------------------------------------------------

    # ---- Binary ----
    if X_bin is not None and vt_bin is not None:
        X_bin = X_bin[model_dict["bin_columns"]]
        X_bin = vt_bin.transform(X_bin)

    # ---- Continuous ----
    if X_cont is not None and vt_cont is not None:

        # 1. Align initial columns
        X_cont = X_cont[model_dict["cont_columns_initial"]]

        # 2. VarianceThreshold
        X_cont = vt_cont.transform(X_cont)

        # 3. Rebuild dataframe
        X_cont = pd.DataFrame(
            X_cont,
            columns=model_dict["cont_after_vt"]
        )

        # 4. Correlation drop
        if model_dict["corr_drop_cols"] is not None:
            X_cont = X_cont.drop(columns=model_dict["corr_drop_cols"])

        # 5. Final column order
        X_cont = X_cont[model_dict["cont_final_cols"]]

        # 6. Impute + Scale
        X_cont = imputer.transform(X_cont)
        X_cont = scaler.transform(X_cont)

    # ---- Merge ----
    if X_bin is not None and X_cont is not None:
        X_final = np.hstack([X_bin, X_cont])
    elif X_bin is not None:
        X_final = X_bin
    else:
        X_final = X_cont

    # ---- Mutual Information selection ----
    X_final = X_final[:, selected_idx]

    # ---- Predict ----
    y_pred = model.predict(X_final)
    f1 = f1_score(y_test, y_pred)

    results[name] = f1

# -------------------------------------------------------------------------
# 3. PRINT FINAL RANKING
# -------------------------------------------------------------------------
print("\n================ FINAL RANKING (BEST MODEL TEST F1) ================")

sorted_results = sorted(results.items(), key=lambda x: x[1], reverse=True)

for rank, (name, score) in enumerate(sorted_results, 1):
    print(f"{rank}. {name:20s} | BEST TEST F1 = {score:.4f}")


================ FINAL RANKING (BEST MODEL TEST F1) ================
1. ECFP+Pharm2D         | BEST TEST F1 = 0.7332
2. ECFP                 | BEST TEST F1 = 0.7228
3. ALL                  | BEST TEST F1 = 0.7159
4. ECFP+RDKit           | BEST TEST F1 = 0.7134
5. Pharm2D+RDKit        | BEST TEST F1 = 0.7125
6. Pharm2D              | BEST TEST F1 = 0.7050
7. RDKit                | BEST TEST F1 = 0.7002


## Model 1

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# -------------------------------------------------------------------------
# 1. LOAD FEATURE FILE
# -------------------------------------------------------------------------

df = pd.read_parquet("wildtype egfr_features.parquet")
print(f"Total samples loaded: {len(df)}")

df.columns = [
    col.replace("ECFP6_", "ECFP_") if col.startswith("ECFP6_") else col
    for col in df.columns
]

df.columns = [
    col.replace("Pharm2D_", "Pharm_") if col.startswith("Pharm2D_") else col
    for col in df.columns
]

assert "y" in df.columns, "Column y not found!"
assert "pIC50 (M)" in df.columns, "Column pIC50 (M) not found!"

# -------------------------------------------------------------------------
# 3. SELECT FINGERPRINT COLUMNS
# -------------------------------------------------------------------------

ecfp_cols = [c for c in df.columns if c.startswith("ECFP_")]
pharm_cols = [c for c in df.columns if c.startswith("Pharm_")]

print(f"ECFP features   : {len(ecfp_cols)}")
print(f"Pharm2D features: {len(pharm_cols)}")

feature_cols = ecfp_cols + pharm_cols

X = df[feature_cols].copy()
y = df["y"].copy()

print(f"Total fingerprint features used: {X.shape[1]}")

# -----------------------------------------------------------------------------
# 4. RECREATE EXACT TRAIN / TEST SPLIT
# -----------------------------------------------------------------------------
X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp,
    test_size=0.125,  # 0.125 × 0.8 = 0.10
    random_state=42,
    stratify=y_tmp
)

print("Recreated test shape:", X_test.shape)

# -----------------------------------------------------------------------------
# 5. LOAD SAVED MODEL PACKAGE
# -----------------------------------------------------------------------------

model_package = joblib.load("model_1.pkl")

model = model_package["model"]
vt = model_package["variance_selector"]
selected_idx = model_package["selected_idx"]

# -----------------------------------------------------------------------------
# 6. APPLY FEATURE SELECTION (IDENTICAL TO TRAINING)
# -----------------------------------------------------------------------------

X_test_vt = vt.transform(X_test)

if selected_idx is not None:
    X_test_final = X_test_vt[:, selected_idx]
else:
    X_test_final = X_test_vt

print("Final test shape:", X_test_final.shape)
print("Model expects:", model.n_features_in_)

assert model.n_features_in_ == X_test_final.shape[1], \
    "Feature mismatch!"

# -----------------------------------------------------------------------------
# 7. PREDICT
# -----------------------------------------------------------------------------

y_pred = model.predict(X_test_final)
y_prob = model.predict_proba(X_test_final)[:, 1]

# -----------------------------------------------------------------------------
# 8. EVALUATE
# -----------------------------------------------------------------------------

print("\n=== Classification Report (TEST SET) ===")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Less potent", "Potent"],
    digits=4
))
print("F1-score:", f1_score(y_test, y_pred))

Total samples loaded: 5276
ECFP features   : 2048
Pharm2D features: 39972
Total fingerprint features used: 42020
Recreated test shape: (1056, 42020)
Final test shape: (1056, 1509)
Model expects: 1509

=== Classification Report (TEST SET) ===
              precision    recall  f1-score   support

 Less potent     0.8900    0.8670    0.8784       737
      Potent     0.7101    0.7524    0.7306       319

    accuracy                         0.8324      1056
   macro avg     0.8000    0.8097    0.8045      1056
weighted avg     0.8356    0.8324    0.8337      1056

F1-score: 0.730593607305936


## Model 2

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# -------------------------------------------------------------------------
# 1. LOAD FEATURE FILE
# -------------------------------------------------------------------------

df = pd.read_parquet("wildtype egfr_features.parquet")
print(f"Total samples loaded: {len(df)}")

df.columns = [
    col.replace("ECFP6_", "ECFP_") if col.startswith("ECFP6_") else col
    for col in df.columns
]

df.columns = [
    col.replace("Pharm2D_", "Pharm_") if col.startswith("Pharm2D_") else col
    for col in df.columns
]

assert "y" in df.columns, "Column y not found!"
assert "pIC50 (M)" in df.columns, "Column pIC50 (M) not found!"

# -------------------------------------------------------------------------
# 2. REMOVE GREY ZONE (pIC50 in [7.2, 7.8])
# -------------------------------------------------------------------------

before_gz = len(df)

df = df[~df["pIC50 (M)"].between(7.2, 7.8)]

after_gz = len(df)

print(f"Removed grey zone samples: {before_gz - after_gz}")
print(f"Remaining samples: {after_gz}")

# -------------------------------------------------------------------------
# 3. SELECT FINGERPRINT COLUMNS
# -------------------------------------------------------------------------

ecfp_cols = [c for c in df.columns if c.startswith("ECFP_")]
pharm_cols = [c for c in df.columns if c.startswith("Pharm_")]

print(f"ECFP features   : {len(ecfp_cols)}")
print(f"Pharm2D features: {len(pharm_cols)}")

feature_cols = ecfp_cols + pharm_cols

X = df[feature_cols].copy()
y = df["y"].copy()

print(f"Total fingerprint features used: {X.shape[1]}")

# -----------------------------------------------------------------------------
# 4. RECREATE EXACT TRAIN / TEST SPLIT
# -----------------------------------------------------------------------------
X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp,
    test_size=0.125,  # 0.125 × 0.8 = 0.10
    random_state=42,
    stratify=y_tmp
)

print("Recreated test shape:", X_test.shape)

# -----------------------------------------------------------------------------
# 5. LOAD SAVED MODEL PACKAGE
# -----------------------------------------------------------------------------

model_package = joblib.load("model_2.pkl")

model = model_package["model"]
vt = model_package["variance_selector"]
selected_idx = model_package["selected_idx"]

# -----------------------------------------------------------------------------
# 6. APPLY FEATURE SELECTION (IDENTICAL TO TRAINING)
# -----------------------------------------------------------------------------

X_test_vt = vt.transform(X_test)

if selected_idx is not None:
    X_test_final = X_test_vt[:, selected_idx]
else:
    X_test_final = X_test_vt

print("Final test shape:", X_test_final.shape)
print("Model expects:", model.n_features_in_)

assert model.n_features_in_ == X_test_final.shape[1], \
    "Feature mismatch!"

# -----------------------------------------------------------------------------
# 7. PREDICT
# -----------------------------------------------------------------------------

y_pred = model.predict(X_test_final)
y_prob = model.predict_proba(X_test_final)[:, 1]

# -----------------------------------------------------------------------------
# 8. EVALUATE
# -----------------------------------------------------------------------------

print("\n=== Classification Report (TEST SET) ===")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Less potent", "Potent"],
    digits=4
))
print("F1-score:", f1_score(y_test, y_pred))

Total samples loaded: 5276
Removed grey zone samples: 775
Remaining samples: 4501
ECFP features   : 2048
Pharm2D features: 39972
Total fingerprint features used: 42020
Recreated test shape: (901, 42020)
Final test shape: (901, 1518)
Model expects: 1518

=== Classification Report (TEST SET) ===
              precision    recall  f1-score   support

 Less potent     0.9198    0.9058    0.9127       658
      Potent     0.7549    0.7860    0.7702       243

    accuracy                         0.8735       901
   macro avg     0.8373    0.8459    0.8414       901
weighted avg     0.8753    0.8735    0.8743       901

F1-score: 0.7701612903225806


## Model 3

In [1]:
# =============================================================================
# REPRODUCE FINAL TEST RESULT FROM SAVED PIPELINE (Model 3)
# =============================================================================

import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

# -----------------------------------------------------------------------------
# 1. LOAD FEATURE FILE
# -----------------------------------------------------------------------------

df = pd.read_parquet("wildtype egfr_features.parquet")
print(f"Total samples loaded: {len(df)}")

assert "y" in df.columns
assert "pIC50 (M)" in df.columns

# -----------------------------------------------------------------------------
# 2. REMOVE GREY ZONE
# -----------------------------------------------------------------------------

df = df[~df["pIC50 (M)"].between(7.2, 7.8)]

# -----------------------------------------------------------------------------
# 3. SELECT FEATURES
# -----------------------------------------------------------------------------

ecfp_cols = [c for c in df.columns if c.startswith("ECFP_")]
pharm_cols = [c for c in df.columns if c.startswith("Pharm_")]

feature_cols = ecfp_cols + pharm_cols

X = df[feature_cols]
y = df["y"]

# -----------------------------------------------------------------------------
# 4. RECREATE EXACT TRAIN / TEST SPLIT
# -----------------------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Test shape:", X_test.shape)

# -----------------------------------------------------------------------------
# 5. LOAD PIPELINE
# -----------------------------------------------------------------------------

pipeline = joblib.load("model_3.pkl")

print("Pipeline loaded successfully.")
print("Model expects:", pipeline.n_features_in_)

# -----------------------------------------------------------------------------
# 6. PREDICT (pipeline tự xử lý feature selection bên trong)
# -----------------------------------------------------------------------------

y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

# -----------------------------------------------------------------------------
# 7. EVALUATE
# -----------------------------------------------------------------------------

print("\n=== Classification Report (TEST SET) ===")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Less potent", "Potent"],
    digits=4
)) 

print("F1-score:", f1_score(y_test, y_pred))

Total samples loaded: 5276
Test shape: (901, 42020)
Pipeline loaded successfully.
Model expects: 42020

=== Classification Report (TEST SET) ===
              precision    recall  f1-score   support

 Less potent     0.9258    0.9103    0.9180       658
      Potent     0.7677    0.8025    0.7847       243

    accuracy                         0.8812       901
   macro avg     0.8468    0.8564    0.8514       901
weighted avg     0.8832    0.8812    0.8821       901

F1-score: 0.7847082494969819


In [ ]:
import joblib
import pandas as pd

# Load model
model = joblib.load("model_3.pkl")

# Load screening features
screen_df = pd.read_parquet("gvae_screening_features.parquet")

# Load training feature columns
train_df = pd.read_parquet("wildtype egfr_features.parquet")
ecfp_cols = [c for c in train_df.columns if c.startswith("ECFP_")]
pharm_cols = [c for c in train_df.columns if c.startswith("Pharm_")]
train_features = ecfp_cols + pharm_cols

# Align feature space
X_screen = screen_df.reindex(columns=train_features, fill_value=0)

# Predict probability
proba = model.predict_proba(X_screen)[:, 1]

screen_df["Potent_probability"] = proba

# Keep only metadata + prediction
cols_to_keep = ["SMILES", "Potent_probability"]

if "BATCH" in screen_df.columns:
    cols_to_keep.append("BATCH")

if "InChIKey" in screen_df.columns:
    cols_to_keep.append("InChIKey")
    
top_hits = screen_df.sort_values("Potent_probability", ascending=False)
output_df = top_hits[cols_to_keep]

# Export
output_df.to_csv("virtual_screening_results.csv", index=False)

print("Saved: virtual_screening_results.csv")

In [ ]:
import pandas as pd

# Đọc CSV (không phải excel)
df = pd.read_csv("virtual_screening_results.csv")

df.head()

from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display

# Sort đúng tên cột
top30 = df.sort_values(
    "Potent_probability",
    ascending=False
).head(30)

mols = [Chem.MolFromSmiles(smi) for smi in top30["SMILES"]]

legends = [
    f"P:{row.Potent_probability:.3f}"
    for _, row in top30.iterrows()
]

img = Draw.MolsToGridImage(
    mols,
    molsPerRow=5,
    subImgSize=(300, 300),
    legends=legends
)

display(img)